In [4]:
import os
import re
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.integrate import odeint
from scipy.optimize import minimize


# =========================
# НАСТРОЙКИ ДЛЯ JUPYTER
# =========================
REGIONS_DIR = "Регионы"
POPULATION_FILE = "население.txt"

TRAIN_WINDOW = 5
PREDICT_WINDOW = 5
diag = 0.60
START_DAY = None          # None = определить автоматически

# Списки для перебора окон
TRAIN_WINDOW_lst = [3, 5, 7, 10, 15, 20, 25, 30]
PREDICT_WINDOW_lst = [20, 25, 30]


# Базовая строка оставлена для совместимости, но для пакетного запуска
# итоговая папка будет собираться динамически внутри run_all_regions(...)
OUTPUT_DIR = "Результаты_IMM_EKF_T10" + str(diag) + "_" + str(TRAIN_WINDOW) + "_" + str(PREDICT_WINDOW)

ACTIVE_COL = "Активные случаи"
RECOVERED_COL = "Выздоровевшие"
DEATH_COL = "Смертельные случаи"


def load_populations(txt_file: str = "население.txt") -> Dict[str, int]:
    populations = {}
    pattern = re.compile(r"^(.*?)\.xlsx\s*-\s*Численность населения:\s*(\d+)\s*$")

    with open(txt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            match = pattern.match(line)
            if match:
                base_name = match.group(1).strip()
                population = int(match.group(2))
                populations[base_name] = population

    if not populations:
        raise ValueError(f"Не удалось прочитать данные из файла {txt_file}")

    return populations


def auto_detect_min_day(data: pd.DataFrame) -> int:
    active = data[ACTIVE_COL].fillna(0)
    recovered = data[RECOVERED_COL].fillna(0)
    both_nonzero = (active > 0) & (recovered > 0)
    return int(both_nonzero.idxmax() + 1 if both_nonzero.any() else 1)


def last_non_zero_day(series: pd.Series) -> int:
    nz = series.fillna(0) != 0
    if not nz.any():
        return len(series)
    return int(nz[nz].index[-1] + 1)


def auto_detect_max_day(data: pd.DataFrame) -> int:
    last_s = last_non_zero_day(data["S"])
    last_i = last_non_zero_day(data[ACTIVE_COL])
    last_r = last_non_zero_day(data["R"])
    return int(min(last_s, last_i, last_r))


def prepare_data(base_name: str, regions_dir: str, population: int) -> Tuple[pd.DataFrame, int]:
    file_path = os.path.join(regions_dir, f"{base_name}.xlsx")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Не найден файл региона: {file_path}")

    data = pd.read_excel(file_path)
    data[ACTIVE_COL] = data[ACTIVE_COL].fillna(0)
    data[RECOVERED_COL] = data[RECOVERED_COL].fillna(0)
    data[DEATH_COL] = data[DEATH_COL].fillna(0)
    data["R"] = data[RECOVERED_COL] + data[DEATH_COL]
    data["S"] = population - data[ACTIVE_COL] - data["R"]
    return data, population


@dataclass
class FittedModel:
    name: str
    params: np.ndarray


class CandidateModel:
    name: str = "BASE"
    param_init: Sequence[float] = ()
    bounds: Sequence[Tuple[float, float]] = ()

    def fit(self, train_data: pd.DataFrame, population: int) -> FittedModel:
        raise NotImplementedError

    def predict(
        self,
        fitted: FittedModel,
        y0: Sequence[float],
        days: int,
        population: int,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        raise NotImplementedError

    def ekf_dynamics(self, x: np.ndarray, dt: float, population: int, fitted: FittedModel) -> np.ndarray:
        raise NotImplementedError

    def ekf_jacobian(self, x: np.ndarray, dt: float, population: int, fitted: FittedModel) -> np.ndarray:
        raise NotImplementedError


class SIRCandidate(CandidateModel):
    name = "SIR"
    param_init = (0.02, 0.02)
    bounds = ((0, 1), (0, 1))

    @staticmethod
    def ode(y, t, n, beta, gamma):
        s, i, r = y
        return -beta * s * i / n, beta * s * i / n - gamma * i, gamma * i

    def fit(self, train_data, population):
        def loss(params):
            beta, gamma = params
            y0 = [
                train_data["S"].iloc[0],
                train_data[ACTIVE_COL].iloc[0],
                train_data["R"].iloc[0],
            ]
            sol = odeint(self.ode, y0, range(len(train_data)), args=(population, beta, gamma))
            s, i, r = sol.T
            return (
                np.mean((s - train_data["S"].values) ** 2)
                + np.mean((i - train_data[ACTIVE_COL].values) ** 2)
                + np.mean((r - train_data["R"].values) ** 2)
            )

        res = minimize(loss, self.param_init, method="L-BFGS-B", bounds=self.bounds)
        return FittedModel(self.name, res.x)

    def predict(self, fitted, y0, days, population):
        beta, gamma = fitted.params
        sol = odeint(self.ode, y0, np.arange(0, days), args=(population, beta, gamma))
        s, i, r = sol.T
        return s, i, r

    def ekf_dynamics(self, x, dt, population, fitted):
        beta, gamma = fitted.params
        s, i, z, r = x
        ds = -beta * s * i / population
        di = beta * s * i / population - gamma * i
        dz = 0.0
        dr = gamma * i
        return np.array([s + ds * dt, i + di * dt, z + dz * dt, r + dr * dt], dtype=float)

    def ekf_jacobian(self, x, dt, population, fitted):
        beta, gamma = fitted.params
        s, i, _, _ = x
        beta_i = beta * i / population
        beta_s = beta * s / population
        j = np.eye(4)
        j[0, 0] += -beta_i * dt
        j[0, 1] += -beta_s * dt
        j[1, 0] += beta_i * dt
        j[1, 1] += (beta_s - gamma) * dt
        j[3, 1] += gamma * dt
        return j


class SEIRCandidate(CandidateModel):
    name = "SEIR"
    param_init = (0.02, 0.02, 0.1)
    bounds = ((0, 1), (0, 1), (0, 1))

    @staticmethod
    def ode(y, t, n, beta, gamma, sigma):
        s, e, i, r = y
        ds = -beta * s * i / n
        de = beta * s * i / n - sigma * e
        di = sigma * e - gamma * i
        dr = gamma * i
        return ds, de, di, dr

    def fit(self, train_data, population):
        def loss(params):
            beta, gamma, sigma = params
            y0 = [
                train_data["S"].iloc[0],
                0.0,
                train_data[ACTIVE_COL].iloc[0],
                train_data["R"].iloc[0],
            ]
            sol = odeint(self.ode, y0, range(len(train_data)), args=(population, beta, gamma, sigma))
            s, _, i, r = sol.T
            return (
                np.mean((s - train_data["S"].values) ** 2)
                + np.mean((i - train_data[ACTIVE_COL].values) ** 2)
                + np.mean((r - train_data["R"].values) ** 2)
            )

        res = minimize(loss, self.param_init, method="L-BFGS-B", bounds=self.bounds)
        return FittedModel(self.name, res.x)

    def predict(self, fitted, y0, days, population):
        beta, gamma, sigma = fitted.params
        s0, i0, r0 = y0
        sol = odeint(self.ode, [s0, 0.0, i0, r0], np.arange(0, days), args=(population, beta, gamma, sigma))
        s, _, i, r = sol.T
        return s, i, r

    def ekf_dynamics(self, x, dt, population, fitted):
        beta, gamma, sigma = fitted.params
        s, i, e, r = x
        ds = -beta * s * i / population
        de = beta * s * i / population - sigma * e
        di = sigma * e - gamma * i
        dr = gamma * i
        return np.array([s + ds * dt, i + di * dt, e + de * dt, r + dr * dt], dtype=float)

    def ekf_jacobian(self, x, dt, population, fitted):
        beta, gamma, sigma = fitted.params
        s, i, _, _ = x
        beta_i = beta * i / population
        beta_s = beta * s / population
        j = np.eye(4)
        j[0, 0] += -beta_i * dt
        j[0, 1] += -beta_s * dt
        j[1, 1] += -gamma * dt
        j[1, 2] += sigma * dt
        j[2, 0] += beta_i * dt
        j[2, 1] += beta_s * dt
        j[2, 2] += -sigma * dt
        j[3, 1] += gamma * dt
        return j


class SIQRCandidate(CandidateModel):
    name = "SIQR"
    param_init = (0.02, 0.02, 0.05, 0.1)
    bounds = ((0, 1), (0, 1), (0, 1), (0, 1))

    @staticmethod
    def ode(y, t, n, beta, gamma, delta, rho):
        s, i, q, r = y
        ds = -beta * s * i / n
        di = beta * s * i / n - gamma * i - delta * i
        dq = delta * i - rho * q
        dr = gamma * i + rho * q
        return ds, di, dq, dr

    def fit(self, train_data, population):
        def loss(params):
            beta, gamma, delta, rho = params
            y0 = [
                train_data["S"].iloc[0],
                train_data[ACTIVE_COL].iloc[0],
                0.0,
                train_data["R"].iloc[0],
            ]
            sol = odeint(self.ode, y0, range(len(train_data)), args=(population, beta, gamma, delta, rho))
            s, i, _, r = sol.T
            return (
                np.mean((s - train_data["S"].values) ** 2)
                + np.mean((i - train_data[ACTIVE_COL].values) ** 2)
                + np.mean((r - train_data["R"].values) ** 2)
            )

        res = minimize(loss, self.param_init, method="L-BFGS-B", bounds=self.bounds)
        return FittedModel(self.name, res.x)

    def predict(self, fitted, y0, days, population):
        beta, gamma, delta, rho = fitted.params
        s0, i0, r0 = y0
        sol = odeint(self.ode, [s0, i0, 0.0, r0], np.arange(0, days), args=(population, beta, gamma, delta, rho))
        s, i, _, r = sol.T
        return s, i, r

    def ekf_dynamics(self, x, dt, population, fitted):
        beta, gamma, delta, rho = fitted.params
        s, i, q, r = x
        ds = -beta * s * i / population
        di = beta * s * i / population - gamma * i - delta * i
        dq = delta * i - rho * q
        dr = gamma * i + rho * q
        return np.array([s + ds * dt, i + di * dt, q + dq * dt, r + dr * dt], dtype=float)

    def ekf_jacobian(self, x, dt, population, fitted):
        beta, gamma, delta, rho = fitted.params
        s, i, _, _ = x
        beta_i = beta * i / population
        beta_s = beta * s / population
        j = np.eye(4)
        j[0, 0] += -beta_i * dt
        j[0, 1] += -beta_s * dt
        j[1, 0] += beta_i * dt
        j[1, 1] += (beta_s - gamma - delta) * dt
        j[2, 1] += delta * dt
        j[2, 2] += -rho * dt
        j[3, 1] += gamma * dt
        j[3, 2] += rho * dt
        return j


class SIRSCandidate(CandidateModel):
    name = "SIRS"
    param_init = (0.02, 0.02, 0.01)
    bounds = ((0, 1), (0, 1), (0, 1))

    @staticmethod
    def ode(y, t, n, beta, gamma, xi):
        s, i, r = y
        ds = -beta * s * i / n + xi * r
        di = beta * s * i / n - gamma * i
        dr = gamma * i - xi * r
        return ds, di, dr

    def fit(self, train_data, population):
        def loss(params):
            beta, gamma, xi = params
            y0 = [
                train_data["S"].iloc[0],
                train_data[ACTIVE_COL].iloc[0],
                train_data["R"].iloc[0],
            ]
            sol = odeint(self.ode, y0, range(len(train_data)), args=(population, beta, gamma, xi))
            s, i, r = sol.T
            return (
                np.mean((s - train_data["S"].values) ** 2)
                + np.mean((i - train_data[ACTIVE_COL].values) ** 2)
                + np.mean((r - train_data["R"].values) ** 2)
            )

        res = minimize(loss, self.param_init, method="L-BFGS-B", bounds=self.bounds)
        return FittedModel(self.name, res.x)

    def predict(self, fitted, y0, days, population):
        beta, gamma, xi = fitted.params
        sol = odeint(self.ode, y0, np.arange(0, days), args=(population, beta, gamma, xi))
        s, i, r = sol.T
        return s, i, r

    def ekf_dynamics(self, x, dt, population, fitted):
        beta, gamma, xi = fitted.params
        s, i, z, r = x
        ds = -beta * s * i / population + xi * r
        di = beta * s * i / population - gamma * i
        dr = gamma * i - xi * r
        return np.array([s + ds * dt, i + di * dt, z, r + dr * dt], dtype=float)

    def ekf_jacobian(self, x, dt, population, fitted):
        beta, gamma, xi = fitted.params
        s, i, _, _ = x
        beta_i = beta * i / population
        beta_s = beta * s / population
        j = np.eye(4)
        j[0, 0] += -beta_i * dt
        j[0, 1] += -beta_s * dt
        j[0, 3] += xi * dt
        j[1, 0] += beta_i * dt
        j[1, 1] += (beta_s - gamma) * dt
        j[3, 1] += gamma * dt
        j[3, 3] += -xi * dt
        return j


class SIRDDECandidate(CandidateModel):
    name = "SIR_DDE"
    param_init = (0.02, 0.02, 0.001)
    bounds = ((0, 1), (0, 1), (0, 0.1))

    def simulate(self, params, y0, days, population, tau=14):
        beta, gamma, nu = params
        s, i, r = y0
        tau = int(tau)
        hist_s = [s] * tau
        s_tr, i_tr, r_tr = [], [], []
        for _ in range(days):
            s_tau = hist_s[-tau] if len(hist_s) >= tau else hist_s[0]
            ds = -beta * s * i / population - nu * s_tau
            di = beta * s * i / population - gamma * i
            dr = gamma * i + nu * s_tau
            s = max(0.0, s + ds)
            i = max(0.0, i + di)
            r = max(0.0, r + dr)
            hist_s.append(s)
            s_tr.append(s)
            i_tr.append(i)
            r_tr.append(r)
        return np.array(s_tr), np.array(i_tr), np.array(r_tr)

    def fit(self, train_data, population):
        tau = 7

        def loss(params):
            y0 = [
                train_data["S"].iloc[0],
                max(train_data[ACTIVE_COL].iloc[0], 1.0),
                train_data["R"].iloc[0],
            ]
            s, i, r = self.simulate(params, y0, len(train_data), population, tau=tau)
            return (
                np.mean((s - train_data["S"].values) ** 2)
                + np.mean((i - train_data[ACTIVE_COL].values) ** 2)
                + np.mean((r - train_data["R"].values) ** 2)
            )

        res = minimize(loss, self.param_init, method="L-BFGS-B", bounds=self.bounds)
        return FittedModel(self.name, res.x)

    def predict(self, fitted, y0, days, population):
        return self.simulate(fitted.params, y0, days, population, tau=14)

    def ekf_dynamics(self, x, dt, population, fitted):
        beta, gamma, nu = fitted.params
        s, i, s_delay, r = x
        ds = -beta * s * i / population - nu * s_delay
        di = beta * s * i / population - gamma * i
        dsd = (s - s_delay) / max(dt, 1e-6)
        dr = gamma * i + nu * s_delay
        return np.array([s + ds * dt, i + di * dt, s_delay + dsd * dt, r + dr * dt], dtype=float)

    def ekf_jacobian(self, x, dt, population, fitted):
        beta, gamma, nu = fitted.params
        s, i, _, _ = x
        beta_i = beta * i / population
        beta_s = beta * s / population
        j = np.eye(4)
        j[0, 0] += -beta_i * dt
        j[0, 1] += -beta_s * dt
        j[0, 2] += -nu * dt
        j[1, 0] += beta_i * dt
        j[1, 1] += (beta_s - gamma) * dt
        j[2, 0] += 1.0
        j[2, 2] += -1.0
        j[3, 1] += gamma * dt
        j[3, 2] += nu * dt
        return j


MODELS: List[CandidateModel] = [
    SIRCandidate(),
    SEIRCandidate(),
    SIQRCandidate(),
    SIRSCandidate(),
    SIRDDECandidate(),
]


def run_imm_probabilities(
    fitted_models: List[FittedModel],
    train_data: pd.DataFrame,
    population: int,
    dt: float = 1.0,
) -> np.ndarray:
    m = len(fitted_models)
    diag_w = diag
    off_w = (1.0 - diag_w) / (m - 1) if m > 1 else 0.0
    pi = np.full((m, m), off_w, dtype=float)
    np.fill_diagonal(pi, diag_w)

    mu = np.ones(m, dtype=float) / m
    x = np.zeros((m, 4), dtype=float)
    p = np.zeros((m, 4, 4), dtype=float)
    q_proc = np.diag([100.0, 40, 10, 5]) * dt   # шум процесса (модель) [S, I, Z, R]
    h_obs = np.array([[0.5, 1, 0.1, 0.3]], dtype=float)  # матрица наблюдения [S, I, Z, R]
    r_meas = np.array([[50.0]], dtype=float)  # шум измерения

    s0 = float(train_data["S"].iloc[0])
    i0 = float(train_data[ACTIVE_COL].iloc[0])
    r0 = float(train_data["R"].iloc[0])
    for i in range(m):
        x[i] = np.array([s0, i0, 0.0, r0], dtype=float)
        p[i] = np.diag([200.0, 50.0, 10.0, 10.0])

    model_map: Dict[str, CandidateModel] = {model.name: model for model in MODELS}
    eps = 1e-12

    for t in range(1, len(train_data)):
        y = float(train_data[ACTIVE_COL].iloc[t])
        c = np.zeros(m, dtype=float)
        x0 = np.zeros((m, 4), dtype=float)
        p0 = np.zeros((m, 4, 4), dtype=float)

        for i in range(m):
            c[i] = np.sum(pi[:, i] * mu)
            if c[i] < eps:
                continue
            mu_ji = (pi[:, i] * mu) / c[i]
            x0[i] = np.sum(mu_ji[:, None] * x, axis=0)
            for j in range(m):
                diff = (x[j] - x0[i]).reshape(-1, 1)
                p0[i] += mu_ji[j] * (p[j] + diff @ diff.T)

        lam = np.zeros(m, dtype=float)
        for i in range(m):
            fitted = fitted_models[i]
            model = model_map[fitted.name]
            x_pred = model.ekf_dynamics(x0[i], dt, population, fitted)
            j = model.ekf_jacobian(x0[i], dt, population, fitted)
            p_pred = j @ p0[i] @ j.T + q_proc

            y_pred = float((h_obs @ x_pred)[0])
            nu = y - y_pred
            s_cov = float((h_obs @ p_pred @ h_obs.T)[0, 0] + r_meas[0, 0])
            k = p_pred @ h_obs.T / max(s_cov, eps)

            x[i] = x_pred + k.flatten() * nu
            p[i] = (np.eye(4) - k @ h_obs) @ p_pred
            x[i] = np.maximum(x[i], 0.0)
            lam[i] = (1.0 / np.sqrt(2 * np.pi * max(s_cov, eps))) * np.exp(-0.5 * nu**2 / max(s_cov, eps))

        mu = c * lam
        mu_sum = float(np.sum(mu))
        if mu_sum < eps:
            mu = np.ones(m, dtype=float) / m
        else:
            mu /= mu_sum

    return mu


def rolling_imm_forecast(
    data: pd.DataFrame,
    population: int,
    start_day: int,
    train_window: int,
    predict_window: int,
) -> pd.DataFrame:
    end_day = auto_detect_max_day(data)
    end_day = max(end_day, start_day + train_window + predict_window - 1)
    subset = data.iloc[start_day - 1: end_day].copy()
    num_windows = len(subset) - train_window - predict_window + 1
    step = predict_window

    result = subset.copy()
    result["Прогноз S"] = np.nan
    result["Прогноз I"] = np.nan
    result["Прогноз R"] = np.nan
    result["Модель прогноза"] = ""
    result["Вероятность модели"] = np.nan

    for i in range(max(0, num_windows)):
        train_start = i * step
        train_end = train_start + train_window
        pred_start = train_end
        pred_end = pred_start + predict_window
        if pred_end > len(subset):
            break

        train = subset.iloc[train_start:train_end]
        pred = subset.iloc[pred_start:pred_end]

        fitted_models = [m.fit(train, population) for m in MODELS]
        mu = run_imm_probabilities(fitted_models, train, population)
        best_idx = int(np.argmax(mu))
        best_model = MODELS[best_idx]
        best_fit = fitted_models[best_idx]

        y0 = [
            train["S"].iloc[-1],
            train[ACTIVE_COL].iloc[-1],
            train["R"].iloc[-1],
        ]
        s_pred, i_pred, r_pred = best_model.predict(best_fit, y0, predict_window, population)

        idx = pred.index
        result.loc[idx, "Прогноз S"] = s_pred
        result.loc[idx, "Прогноз I"] = i_pred
        result.loc[idx, "Прогноз R"] = r_pred
        result.loc[idx, "Модель прогноза"] = best_fit.name
        result.loc[idx, "Вероятность модели"] = float(mu[best_idx])

    result.rename(columns={"S": "Факт S", ACTIVE_COL: "Факт I", "R": "Факт R"}, inplace=True)
    result["% откл S"] = np.where(
        result["Факт S"] > 0,
        np.abs(result["Прогноз S"] - result["Факт S"]) / result["Факт S"] * 100,
        0,
    )
    result["% откл I"] = np.where(
        result["Факт I"] > 0,
        np.abs(result["Прогноз I"] - result["Факт I"]) / result["Факт I"] * 100,
        0,
    )
    result["% откл R"] = np.where(
        result["Факт R"] > 0,
        np.abs(result["Прогноз R"] - result["Факт R"]) / result["Факт R"] * 100,
        0,
    )

    if "Дата" not in result.columns:
        result["Дата"] = pd.date_range("2020-01-01", periods=len(result))

    return result[
        [
            "Дата",
            "Факт S",
            "Факт I",
            "Факт R",
            "Прогноз S",
            "Прогноз I",
            "Прогноз R",
            "% откл S",
            "% откл I",
            "% откл R",
            "Модель прогноза",
            "Вероятность модели",
        ]
    ]


def run_all_regions(
    regions_dir=REGIONS_DIR,
    population_file=POPULATION_FILE,
    train_window=TRAIN_WINDOW,
    predict_window=PREDICT_WINDOW,
    start_day=START_DAY,
    output_dir=None,
):
    if output_dir is None:
        output_dir = f"Результаты_IMM_EKF_{diag}_{train_window}_{predict_window}"

    populations = load_populations(population_file)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Найдено регионов в population file: {len(populations)}")
    print(f"Папка вывода: {output_dir}")
    print(f"Окна: обучение={train_window}, прогноз={predict_window}, diag={diag}")

    for base_name, population in populations.items():
        try:
            print(f"\nОбработка региона: {base_name}, население={population}")

            data, population = prepare_data(base_name, regions_dir, population)
            min_day = auto_detect_min_day(data) if start_day is None else int(start_day)

            out = rolling_imm_forecast(
                data=data,
                population=population,
                start_day=min_day,
                train_window=train_window,
                predict_window=predict_window,
            )

            end_day = auto_detect_max_day(data)
            out_file = os.path.join(
                output_dir,
                f"Предсказания_{base_name}_{min_day}_{end_day}_IMM_EKF.xlsx",
            )
            if os.path.exists(out_file):
                os.remove(out_file)

            out.to_excel(out_file, index=False)

            print(f"Готово. Min day: {min_day}, End day: {end_day}")
            print(f"Файл: {out_file}")

        except Exception as e:
            print(f"Ошибка при обработке региона {base_name}: {e}")


def run_all_regions_for_window_lists(
    train_window_list,
    predict_window_list,
    regions_dir=REGIONS_DIR,
    population_file=POPULATION_FILE,
    start_day=START_DAY,
):
    for train_window in train_window_list:
        for predict_window in predict_window_list:
            print("\n" + "=" * 90)
            print(f"Запуск для пары окон: обучение={train_window}, прогноз={predict_window}, diag={diag}")
            print("=" * 90)

            run_all_regions(
                regions_dir=regions_dir,
                population_file=population_file,
                train_window=train_window,
                predict_window=predict_window,
                start_day=start_day,
                output_dir=f"Результаты_IMM-EKF_{diag}_{train_window}_{predict_window}",
            )


# Запуск для всех регионов и всех пар окон
run_all_regions_for_window_lists(
    train_window_list=TRAIN_WINDOW_lst,
    predict_window_list=PREDICT_WINDOW_lst,
)


Запуск для пары окон: обучение=3, прогноз=20, diag=0.6
Найдено регионов в population file: 16
Папка вывода: Результаты_IMM-EKF_0.6_3_20
Окна: обучение=3, прогноз=20, diag=0.6

Обработка региона: 056_Beijing, население=21710000
Готово. Min day: 1, End day: 453
Файл: Результаты_IMM-EKF_0.6_3_20\Предсказания_056_Beijing_1_453_IMM_EKF.xlsx

Обработка региона: 058_Berlin, население=3769000
Готово. Min day: 1, End day: 400
Файл: Результаты_IMM-EKF_0.6_3_20\Предсказания_058_Berlin_1_400_IMM_EKF.xlsx

Обработка региона: 070_Brussels, население=1218000
Готово. Min day: 1, End day: 218
Файл: Результаты_IMM-EKF_0.6_3_20\Предсказания_070_Brussels_1_218_IMM_EKF.xlsx

Обработка региона: 109_Ciudad_de_Mexico, население=9209000
Готово. Min day: 1, End day: 394
Файл: Результаты_IMM-EKF_0.6_3_20\Предсказания_109_Ciudad_de_Mexico_1_394_IMM_EKF.xlsx

Обработка региона: 126_Delhi, население=32230000
Готово. Min day: 1, End day: 373
Файл: Результаты_IMM-EKF_0.6_3_20\Предсказания_126_Delhi_1_373_IMM_EKF.xls